In [1]:
import numpy as np

In [2]:
def align_depth_lstsq(pred, gt, mask):

    """

    Подбирает scale и shift через least squares: gt ≈ scale * pred + shift

    Возвращает aligned predicted depth в метрах.

    """

    pred_valid = pred[mask]

    gt_valid = gt[mask]



    A = np.stack([pred_valid, np.ones_like(pred_valid)], axis=1)

    scale, shift = np.linalg.lstsq(A, gt_valid, rcond=None)[0]



    return scale * pred + shift, scale, shift

In [3]:
def compute_depth_metrics(pred, gt, max_depth=80.0):

    """

    Стандартные метрики оценки глубины (KITTI eval protocol)

    """

    mask = (gt > 0.1) & (gt < max_depth)

    pred = pred[mask]

    gt = gt[mask]



    abs_rel = np.mean(np.abs(pred - gt) / gt)

    sq_rel = np.mean((pred - gt) ** 2 / gt)

    rmse = np.sqrt(np.mean((pred - gt) ** 2))

    rmse_log = np.sqrt(np.mean((np.log(pred) - np.log(gt)) ** 2))



    thresh = np.maximum(pred / gt, gt / pred)

    delta1 = np.mean(thresh < 1.25)

    delta2 = np.mean(thresh < 1.25 ** 2)

    delta3 = np.mean(thresh < 1.25 ** 3)



    return {

        'AbsRel': abs_rel,

        'SqRel': sq_rel,

        'RMSE': rmse,

        'RMSE_log': rmse_log,

        'δ < 1.25': delta1,

        'δ < 1.25²': delta2,

        'δ < 1.25³': delta3,

    }

## Загрузка всего и вся

In [ ]:
!gdown --folder --remaining-ok "https://drive.google.com/drive/folders/1-1606rH8lv-vf6kJ-fpH78TMu5bBExwn?usp=drive_link"

Retrieving folder contents
Retrieving folder 1G8W_tpHxmVgVUkCcpLoZ_7gKHdrTrtDr Scene01
Processing file 10US12kjZXBkIfrmy-bKnDL2XsnPJBwf0 00000.npy
Processing file 1tf73CkE4U2O8TO3qtQHwWK15Ml8lsPbZ 00001.npy
Processing file 18cOTHCB6109eekTbgDH2wd83mQzR1nv5 00002.npy
Processing file 1TTESNaeiPTKZuaZbV8ebWOVYME5MuwMp 00003.npy
Processing file 1WQ9rjSj-cFGjopMf2UDTq8CC2rDMQNO7 00004.npy
Processing file 11isWJYnLWrOLjokT_XdF--KntwNASSWG 00005.npy
Processing file 1iAA4UNCYrkNB5Wt80ezpIklSjH89pvE0 00006.npy
Processing file 1emtNgndD8W5Z4rB7HuOz0qMn8k_Ipl4P 00007.npy
Processing file 10IBR7GNyBC4nEt5jYqsOJM3WX_ilDB4C 00008.npy
Processing file 1TN_SGYo5eMp23MwRJDGzUpcN7XpZH9vg 00009.npy
Processing file 10b9KlsCALJXSbq3vH2a-r8eAcNwWxn9G 00010.npy
Processing file 16VKQhgrfhKjGZThUZfZewAx25vhkUI-o 00011.npy
Processing file 1HqBU9UzR_R5zN0JZXM8fcr0drg5m3K37 00012.npy
Processing file 1bWpHqHsd3Qk0KSZshyrxEiao3M436Gmw 00013.npy
Processing file 1A6tRk9-1grOBc73vH3wq3lhfYirbRlRX 00014.npy
Processing fi

In [ ]:
!gdown --id 1kkK-qZO-5hwyybCFuaEvANlRXiv-5xxj

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1kkK-qZO-5hwyybCFuaEvANlRXiv-5xxj

but Gdown can't. Please check connections and permissions.


In [ ]:
!gdown 1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS

Downloading...
From (original): https://drive.google.com/uc?id=1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS
From (redirected): https://drive.google.com/uc?id=1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS&confirm=t&uuid=e38d612f-bdfc-4655-8070-67c93357d852
To: /content/vkitti_2.0.3_rgb.tar
100% 7.53G/7.53G [01:30<00:00, 83.2MB/s]


In [ ]:
import os, tarfile


VKITTI_LOCAL = '/content/vkitti2'

os.makedirs(VKITTI_LOCAL, exist_ok=True)


for archive in ['vkitti_2.0.3_rgb.tar', 'vkitti_2.0.3_depth.tar']:

    archive_path = f'/content/{archive}'

    if os.path.exists(archive_path):


        print(f'📦 Распаковываю {archive}...')

        with tarfile.open(archive_path, 'r') as tar:

            tar.extractall(VKITTI_LOCAL)

        print(f'   ✅ Готово')

    else:

        print(f'   ❌ Не найден: {archive}')

📦 Распаковываю vkitti_2.0.3_rgb.tar...


/tmp/ipykernel_23051/3399381077.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(VKITTI_LOCAL)


   ✅ Готово
📦 Распаковываю vkitti_2.0.3_depth.tar...
   ✅ Готово


In [27]:
!gdown --id  1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8
From (redirected): https://drive.google.com/uc?id=1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8&confirm=t&uuid=3a62e478-fc33-41bc-9e4b-2166e5be5d79
To: /content/_loader.py
100% 1.90k/1.90k [00:00<00:00, 10.0MB/s]


In [28]:
from _loader import VKITTI2Loader
VKITTI_RGB = '/content/vkitti2'
loader = VKITTI2Loader(VKITTI_RGB)

## продолжеие

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
VKITTI_DRIVE = '/content/drive/MyDrive/3dcv-project/data/vkitti2'
VKITTI_LOCAL = '/content/vkitti2'

In [8]:
import os
os.makedirs(VKITTI_LOCAL, exist_ok=True)

In [23]:
import  tarfile

for archive in ['vkitti_2.0.3_rgb.tar', 'vkitti_2.0.3_depth.tar']:

    archive_path = f'{VKITTI_DRIVE}/{archive}'

    if os.path.exists(archive_path):

        print(f'📦 Распаковываю {archive}...')

        with tarfile.open(archive_path, 'r') as tar:

            tar.extractall(VKITTI_LOCAL)

        print(f'   ✅ Готово')

    else:

        print(f'   ❌ Не найден: {archive}')



📦 Распаковываю vkitti_2.0.3_rgb.tar...


/tmp/ipykernel_3227/3785276774.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(VKITTI_LOCAL)


   ✅ Готово
📦 Распаковываю vkitti_2.0.3_depth.tar...
   ✅ Готово


In [24]:
!cp -r /content/vkitti2 "/content/drive/MyDrive/3dcv-project/data/vkitti2_unpacked"

In [26]:
!ls "/content/drive/MyDrive/3dcv-project/data/vkitti2_unpacked"

 Scene01   Scene06  'Scene18 (1)'  'Scene20 (1)'
 Scene02   Scene18   Scene20	    vkitti2


In [ ]:

!find {VKITTI_LOCAL} -maxdepth 4 -type d | head -30

In [9]:
!ls

drive  sample_data  vkitti2


In [20]:
!ls /content/vkitti2

In [12]:
DEPTH_PRED_DIR = '/content/drive/MyDrive/3dcv-project/results/track_b/depth_pred_vkitti'

In [13]:

print(os.path.exists(DEPTH_PRED_DIR))

True


In [29]:
import matplotlib.pyplot as plt

loader = VKITTI2Loader('/content/vkitti2')

frames = loader.list_frames('Scene01', 'clone')

print(f'Найдено {len(frames)} кадров в Scene01/clone')

Найдено 447 кадров в Scene01/clone


In [30]:
import pandas as pd
from tqdm.notebook import tqdm


results = []

In [31]:
print(DEPTH_PRED_DIR)

/content/drive/MyDrive/3dcv-project/results/track_b/depth_pred_vkitti


In [32]:
for scene in loader.SCENES:

    frames = loader.list_frames(scene, 'clone')

    for frame_id in tqdm(frames, desc=scene):

        gt_depth = loader.load_depth(scene, 'clone', frame_id)

        pred_path = f'{DEPTH_PRED_DIR}/{scene}/{frame_id}.npy'

        if not os.path.exists(pred_path):

            continue

        pred_relative = np.load(pred_path)



        # Если предсказание и GT разного размера — ресайзим

        if pred_relative.shape != gt_depth.shape:

            from PIL import Image

            pred_relative = np.array(Image.fromarray(pred_relative).resize(

                (gt_depth.shape[1], gt_depth.shape[0])))



        # Калибровка

        mask = (gt_depth > 0.1) & (gt_depth < 80)

        if mask.sum() < 100:

            continue

        pred_metric, _, _ = align_depth_lstsq(pred_relative, gt_depth, mask)



        # Метрики

        m = compute_depth_metrics(pred_metric, gt_depth)

        m['scene'] = scene

        m['frame'] = frame_id

        results.append(m)

df = pd.DataFrame(results)

df.to_csv('/content/drive/MyDrive/3dcv-project/results/track_b/B3_depth_metrics_vkitti.csv',

          index=False)

print('n📊 Сводка по VKITTI2:')

print(df[['AbsRel', 'RMSE', 'δ < 1.25']].mean())

Scene01:   0%|          | 0/447 [00:00<?, ?it/s]

/tmp/ipykernel_3227/1536373973.py:23: RuntimeWarning: invalid value encountered in log
  rmse_log = np.sqrt(np.mean((np.log(pred) - np.log(gt)) ** 2))


Scene02:   0%|          | 0/233 [00:00<?, ?it/s]

Scene06:   0%|          | 0/270 [00:00<?, ?it/s]

Scene18:   0%|          | 0/339 [00:00<?, ?it/s]

Scene20:   0%|          | 0/837 [00:00<?, ?it/s]

n📊 Сводка по VKITTI2:
AbsRel      0.506330
RMSE        9.725141
δ < 1.25    0.390757
dtype: float64


In [37]:
scene_summary = (
    df.groupby('scene')[['AbsRel', 'RMSE', 'δ < 1.25']]
      .mean()
      .round(3)
)

scene_summary.columns = [
    'AbsRel ↓',
    'RMSE ↓',
    'δ < 1.25 ↑'
]

styled = (
    scene_summary.style
    .format('{:.3f}')

    # Лучшие значения
    .highlight_min(
        subset=['AbsRel ↓', 'RMSE ↓'],
        color='#d9ead3'
    )
    .highlight_max(
        subset=['δ < 1.25 ↑'],
        color='#d9ead3'
    )

    # Худшие значения
    .highlight_max(
        subset=['AbsRel ↓', 'RMSE ↓'],
        color='#f4cccc'
    )
    .highlight_min(
        subset=['δ < 1.25 ↑'],
        color='#f4cccc'
    )

    .set_caption("Сводка метрик по сценам VKITTI2")
)

styled

,AbsRel ↓,RMSE ↓,δ < 1.25 ↑
scene,,,
Scene01,0.491,9.233,0.391
Scene02,0.608,11.566,0.353
Scene06,0.495,10.278,0.432
Scene18,0.542,9.992,0.397
Scene20,0.448,8.714,0.381


для AbsRel и RMSE
- маленькое значение → хорошо → зелёное
- большое значение → плохо → красное


для δ < 1.25
- большое значение → хорошо → зелёное
- маленькое значение → плохо → красное

In [38]:
scene_summary = (
    df.groupby('scene')[['AbsRel', 'RMSE', 'δ < 1.25']]
      .mean()
      .round(3)
)

styled = (
    scene_summary.style
    .format({
        'AbsRel': '{:.3f}',
        'RMSE': '{:.3f}',
        'δ < 1.25': '{:.3f}'
    })
    .set_caption("Сводка метрик по сценам VKITTI2")
    .set_table_styles([
        {
            'selector': 'caption',
            'props': [
                ('font-size', '14px'),
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]
        },
        {
            'selector': 'th',
            'props': [
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('text-align', 'center')
            ]
        }
    ])
)

styled

,AbsRel,RMSE,δ < 1.25
scene,,,
Scene01,0.491,9.233,0.391
Scene02,0.608,11.566,0.353
Scene06,0.495,10.278,0.432
Scene18,0.542,9.992,0.397
Scene20,0.448,8.714,0.381


In [33]:
df.to_csv(
    '/content/drive/MyDrive/3dcv-project/results/track_b/B3_depth_metrics_vkitti.csv',
    index=False
)

print("Файл сохранён")

Файл сохранён
